In [0]:
# Reuse employees data from Day 8
emp_data = [
    (1,  "Ravi",   "Engineering", "Pune",      55000, 28),
    (2,  "Priya",  "HR",          "Mumbai",    42000, 32),
    (3,  "Arjun",  "Engineering", "Delhi",     72000, 26),
    (4,  "Sneha",  "Finance",     "Pune",      61000, 30),
    (5,  "Rohit",  "Engineering", "Mumbai",    80000, 35),
    (6,  "Meera",  "HR",          "Bangalore", 39000, 27),
    (7,  "Karan",  "Finance",     "Delhi",     55000, 29),
    (8,  "Divya",  "Engineering", "Pune",      91000, 33),
    (9,  "Nitin",  "HR",          "Mumbai",    44000, 31),
    (10, "Anjali", "Finance",     "Bangalore", 67000, 28),
    (11, "Rahul",  "Engineering", "Delhi",     68000, 30),
    (12, "Pooja",  "HR",          "Pune",      41000, 26),
]
cols = ["emp_id","name","dept","city","salary","age"]
df = spark.createDataFrame(emp_data, cols)
display(df)

emp_id,name,dept,city,salary,age
1,Ravi,Engineering,Pune,55000,28
2,Priya,HR,Mumbai,42000,32
3,Arjun,Engineering,Delhi,72000,26
4,Sneha,Finance,Pune,61000,30
5,Rohit,Engineering,Mumbai,80000,35
6,Meera,HR,Bangalore,39000,27
7,Karan,Finance,Delhi,55000,29
8,Divya,Engineering,Pune,91000,33
9,Nitin,HR,Mumbai,44000,31
10,Anjali,Finance,Bangalore,67000,28


In [0]:
# Register DataFrame as a temp view — give it a SQL name
df.createOrReplaceTempView("employees")

# Now you can write SQL against "employees"
print("✅ Temp view created!")

# Verify it exists
spark.sql("SHOW TABLES").show()

✅ Temp view created!
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|   employee_rankings|      false|
| default|           employees|      false|
| default|     employees_clean|      false|
| default|employees_partiti...|      false|
| default|        high_earners|      false|
| default|     sales_analytics|      false|
|        |           employees|       true|
+--------+--------------------+-----------+



In [0]:
# Just verify spark is available
print(type(spark))
print(spark.version)

# Output:
# <class 'pyspark.sql.session.SparkSession'>
# 4.1.0

<class 'pyspark.sql.connect.session.SparkSession'>
4.1.0


In [0]:
# SELECT — choose columns
spark.sql("SELECT name, dept, salary FROM employees").show()

# WHERE — filter rows
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE salary > 60000
""").show()

# ORDER BY
spark.sql("""
    SELECT name, dept, salary
    FROM employees
    ORDER BY salary DESC
""").show()

# LIMIT
spark.sql("""
    SELECT name, salary
    FROM employees
    ORDER BY salary DESC
    LIMIT 5
""").show()

# DISTINCT
spark.sql("SELECT DISTINCT dept FROM employees").show()

+------+-----------+------+
|  name|       dept|salary|
+------+-----------+------+
|  Ravi|Engineering| 55000|
| Priya|         HR| 42000|
| Arjun|Engineering| 72000|
| Sneha|    Finance| 61000|
| Rohit|Engineering| 80000|
| Meera|         HR| 39000|
| Karan|    Finance| 55000|
| Divya|Engineering| 91000|
| Nitin|         HR| 44000|
|Anjali|    Finance| 67000|
| Rahul|Engineering| 68000|
| Pooja|         HR| 41000|
+------+-----------+------+

+------+------+
|  name|salary|
+------+------+
| Arjun| 72000|
| Sneha| 61000|
| Rohit| 80000|
| Divya| 91000|
|Anjali| 67000|
| Rahul| 68000|
+------+------+

+------+-----------+------+
|  name|       dept|salary|
+------+-----------+------+
| Divya|Engineering| 91000|
| Rohit|Engineering| 80000|
| Arjun|Engineering| 72000|
| Rahul|Engineering| 68000|
|Anjali|    Finance| 67000|
| Sneha|    Finance| 61000|
|  Ravi|Engineering| 55000|
| Karan|    Finance| 55000|
| Nitin|         HR| 44000|
| Priya|         HR| 42000|
| Pooja|         HR| 41000

In [0]:
# GROUP BY + aggregations
spark.sql("""
    SELECT
        dept,
        COUNT(*)           AS headcount,
        ROUND(AVG(salary),0) AS avg_salary,
        SUM(salary)        AS total_salary,
        MAX(salary)        AS max_salary,
        MIN(salary)        AS min_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""").show()

# HAVING — filter after group
spark.sql("""
    SELECT dept, ROUND(AVG(salary),0) AS avg_sal
    FROM employees
    GROUP BY dept
    HAVING avg_sal > 55000
""").show()

+-----------+---------+----------+------------+----------+----------+
|       dept|headcount|avg_salary|total_salary|max_salary|min_salary|
+-----------+---------+----------+------------+----------+----------+
|Engineering|        5|   73200.0|      366000|     91000|     55000|
|    Finance|        3|   61000.0|      183000|     67000|     55000|
|         HR|        4|   41500.0|      166000|     44000|     39000|
+-----------+---------+----------+------------+----------+----------+

+-----------+-------+
|       dept|avg_sal|
+-----------+-------+
|Engineering|73200.0|
|    Finance|61000.0|
+-----------+-------+



In [0]:
# CASE WHEN — exactly like when().otherwise() in PySpark
spark.sql("""
    SELECT
        name,
        salary,
        CASE
            WHEN salary > 70000 THEN 'High'
            WHEN salary > 50000 THEN 'Medium'
            ELSE 'Low'
        END AS salary_grade
    FROM employees
""").show()

+------+------+------------+
|  name|salary|salary_grade|
+------+------+------------+
|  Ravi| 55000|      Medium|
| Priya| 42000|         Low|
| Arjun| 72000|        High|
| Sneha| 61000|      Medium|
| Rohit| 80000|        High|
| Meera| 39000|         Low|
| Karan| 55000|      Medium|
| Divya| 91000|        High|
| Nitin| 44000|         Low|
|Anjali| 67000|      Medium|
| Rahul| 68000|      Medium|
| Pooja| 41000|         Low|
+------+------+------------+



In [0]:
# Create dept view too
dept_data = [(1,"Engineering"),(2,"HR"),(3,"Finance")]
dept_df = spark.createDataFrame(dept_data, ["dept_id","dept_name"])
dept_df.createOrReplaceTempView("departments")

# Register employees with dept_id
emp2 = [(1,"Ravi",1,55000),(2,"Priya",2,42000),
        (3,"Arjun",1,72000),(4,"Sneha",3,61000)]
emp2_df = spark.createDataFrame(emp2,
    ["emp_id","name","dept_id","salary"])
emp2_df.createOrReplaceTempView("emp_with_id")

# JOIN in SQL
spark.sql("""
    SELECT
        e.name,
        e.salary,
        d.dept_name
    FROM emp_with_id e
    INNER JOIN departments d
        ON e.dept_id = d.dept_id
    ORDER BY e.salary DESC
""").show()

+-----+------+-----------+
| name|salary|  dept_name|
+-----+------+-----------+
|Arjun| 72000|Engineering|
|Sneha| 61000|    Finance|
| Ravi| 55000|Engineering|
|Priya| 42000|         HR|
+-----+------+-----------+



In [0]:
# Subquery — employees earning above average
spark.sql("""
    SELECT name, dept, salary
    FROM employees
    WHERE salary > (
        SELECT AVG(salary) FROM employees
    )
    ORDER BY salary DESC
""").show()

# Subquery in FROM clause
spark.sql("""
    SELECT dept, avg_sal
    FROM (
        SELECT dept, ROUND(AVG(salary),0) AS avg_sal
        FROM employees
        GROUP BY dept
    )
    WHERE avg_sal > 55000
""").show()

+------+-----------+------+
|  name|       dept|salary|
+------+-----------+------+
| Divya|Engineering| 91000|
| Rohit|Engineering| 80000|
| Arjun|Engineering| 72000|
| Rahul|Engineering| 68000|
|Anjali|    Finance| 67000|
| Sneha|    Finance| 61000|
+------+-----------+------+

+-----------+-------+
|       dept|avg_sal|
+-----------+-------+
|Engineering|73200.0|
|    Finance|61000.0|
+-----------+-------+



In [0]:
# In production you mix SQL and DataFrame API freely
# SQL for complex queries, DataFrame for programmatic logic

# Step 1 — SQL query → returns DataFrame
top_earners = spark.sql("""
    SELECT name, dept, salary,
           RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dept_rank
    FROM employees
""")

# Step 2 — Apply DataFrame operations on SQL result
result = top_earners \
    .filter(col("dept_rank") == 1) \
    .select("dept", "name", "salary")

display(result)

# Step 3 — Save final result as Delta
spark.sql("DROP TABLE IF EXISTS top_earners_by_dept")
result.write.format("delta").saveAsTable("top_earners_by_dept")
print("✅ Saved!")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6921550549944846>, line 13
      5 top_earners = spark.sql("""
      6     SELECT name, dept, salary,
      7            RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dept_rank
      8     FROM employees
      9 """)
     11 # Step 2 — Apply DataFrame operations on SQL result
     12 result = top_earners \
---> 13     .filter(col("dept_rank") == 1) \
     14     .select("dept", "name", "salary")
     16 display(result)
     18 # Step 3 — Save final result as Delta

NameError: name 'col' is not defined

# In Databricks you can use %sql magic command
# Write SQL directly without spark.sql() wrapper

In [0]:
%sql
SELECT
    dept,
    COUNT(*) AS headcount,
    ROUND(AVG(salary),0) AS avg_salary
FROM employees
GROUP BY dept
ORDER BY headcount DESC

dept,headcount,avg_salary
Engineering,5,73200.0
HR,4,41500.0
Finance,3,61000.0


In [0]:
# %sql result is shown as interactive table
# Great for quick exploration in Databricks